## 1 Setup

### 1.1 Import libraries

In [1]:
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 2.4.1
Pandas version: 3.0.0


### 1.2 Import data

#### Movies Data

In [2]:
mov_df = pd.read_csv("data/movies.csv", index_col=0)
row_count1, col_count1 = mov_df.shape

print(f"Dataset has {row_count1} rows and {col_count1} columns.")

Dataset has 1465 rows and 11 columns.


In [3]:
mov_df.head(3)

,id,budget,popularity,revenue,title,vote_average,vote_count,director_id,year,month,day
0,43597,237000000,150,2787965087,Avatar,7.2,11800,4762,2009,Dec,Thursday
1,43598,300000000,139,961000000,Pirates of the Caribbean: At World's End,6.9,4500,4763,2007,May,Saturday
2,43599,245000000,107,880674609,Spectre,6.3,4466,4764,2015,Oct,Monday


#### Directors Data

In [4]:
dir_df = pd.read_csv("data/directors.csv", index_col=0)
row_count2, col_count2 = dir_df.shape

print(f"Dataset has {row_count2} rows and {col_count2} columns.")

Dataset has 2349 rows and 3 columns.


In [5]:
dir_df.head(3)

,director_name,id,gender
0,James Cameron,4762,Male
1,Gore Verbinski,4763,Male
2,Sam Mendes,4764,Male


## 2 EDA

#### Validate `director_id` in Movies dataset

Get count of Unique directors in directors dataset

In [6]:
mov_df["director_id"].nunique()

199

Get count of Unique directors in directors dataset

In [7]:
dir_df["id"].nunique()

2349

Directors in Movies dataset is subset of directors in Directors dataset i.e., all directors in Directors dataset does not have their movie details in Movies dataset.

In [8]:
mask = mov_df["director_id"].isin(dir_df["id"])
mask

0       True
1       True
2       True
3       True
5       True
        ... 
4736    True
4743    True
4748    True
4749    True
4768    True
Name: director_id, Length: 1465, dtype: bool

In [9]:
mask.value_counts()

director_id
True    1465
Name: count, dtype: int64

or

In [10]:
np.all(mask)

np.True_

#### Get director's name for each movie

In [11]:
full_df = pd.merge(mov_df, dir_df, left_on="director_id", right_on="id", how="inner")

display(full_df.head(3))
print("Shape:", full_df.shape)

,id_x,budget,popularity,revenue,title,vote_average,vote_count,director_id,year,month,day,director_name,id_y,gender
0,43597,237000000,150,2787965087,Avatar,7.2,11800,4762,2009,Dec,Thursday,James Cameron,4762,Male
1,43598,300000000,139,961000000,Pirates of the Caribbean: At World's End,6.9,4500,4763,2007,May,Saturday,Gore Verbinski,4763,Male
2,43599,245000000,107,880674609,Spectre,6.3,4466,4764,2015,Oct,Monday,Sam Mendes,4764,Male


Shape: (1465, 14)


In [12]:
full_df[["title", "director_name"]].head()

,title,director_name
0,Avatar,James Cameron
1,Pirates of the Caribbean: At World's End,Gore Verbinski
2,Spectre,Sam Mendes
3,The Dark Knight Rises,Christopher Nolan
4,Spider-Man 3,Sam Raimi


#### Encode gender column in Directors dataset

##### Before encoding

In [13]:
dir_df["gender"].info()

<class 'pandas.Series'>
RangeIndex: 2349 entries, 0 to 2348
Series name: gender
Non-Null Count  Dtype
--------------  -----
1724 non-null   str  
dtypes: str(1)
memory usage: 18.5 KB


In [14]:
missing_gender = dir_df["gender"].isna()
dir_df[missing_gender]["gender"]

18      NaN
40      NaN
49      NaN
52      NaN
55      NaN
       ... 
2341    NaN
2342    NaN
2343    NaN
2345    NaN
2346    NaN
Name: gender, Length: 625, dtype: str

In [15]:
len(dir_df[missing_gender]["gender"])

625

In [16]:
dir_df["gender"].value_counts()

gender
Male      1574
Female     150
Name: count, dtype: int64

##### After encoding

In [17]:
def encode_gender(value):
    """
    Function to encode gender
    """
    if value == "Male":
        return 1
    elif value == "Female":
        return 0
    else:
        return None


dir_df["gender_en"] = dir_df["gender"].apply(encode_gender)

In [18]:
dir_df["gender_en"].value_counts()

gender_en
1.0    1574
0.0     150
Name: count, dtype: int64

In [19]:
tmp_df = pd.DataFrame(
    [
        [1, 11, None],
        [2, None, None],
        [3, 33, 0],
    ]
)
tmp_df

,0,1,2
0,1,11.0,NaN
1,2,NaN,NaN
2,3,33.0,0.0


In [20]:
tmp_df.apply(sum)  # axis = 0

0    6.0
1    NaN
2    NaN
dtype: float64

In [21]:
tmp_df.apply(sum, axis=1)  # axis = 1

0     NaN
1     NaN
2    36.0
dtype: float64

In [22]:
def is_null(x):
    return sum(x.isnull())


tmp_df.apply(is_null, axis=1)

0    1
1    2
2    0
dtype: int64

In [23]:
# Column wise null count.
tmp_df.isnull().sum(axis=0)

0    0
1    1
2    2
dtype: int64

In [24]:
# Row wise null count.
tmp_df.isnull().sum(axis=1)

0    1
1    2
2    0
dtype: int64

#### Calculate profit

In [25]:
mov_df[["revenue", "budget"]].head(3)

,revenue,budget
0,2787965087,237000000
1,961000000,300000000
2,880674609,245000000


In [26]:
(mov_df["revenue"] - mov_df["budget"]) / 10_000_000

0       255.096509
1        66.100000
2        63.567461
3        83.493910
5        63.287163
           ...    
4736      0.032195
4743      0.312413
4748      0.000000
4749      0.000000
4768      0.182092
Length: 1465, dtype: float64

In [27]:
def get_profit_in_mil(row):
    """
    Function to get profit in million.
    """
    return round((row["revenue"] - row["budget"]) / 10_000_000, 2)


mov_df[["revenue", "budget"]].apply(get_profit_in_mil, axis=1)

0       255.10
1        66.10
2        63.57
3        83.49
5        63.29
         ...  
4736      0.03
4743      0.31
4748      0.00
4749      0.00
4768      0.18
Length: 1465, dtype: float64

In [28]:
tmp = pd.DataFrame(
    data=[
        [1, 11],
        [2, 22],
        [3, 33],
    ],
    columns=["col_1", "col_2"],
)
tmp

,col_1,col_2
0,1,11
1,2,22
2,3,33


In [29]:
tmp[["col_1", "col_2"]].sum(axis=1)

0    12
1    24
2    36
dtype: int64

## 3 `groupby`

### 3.1 Basics

In [30]:
mov_df.head()

,id,budget,popularity,revenue,title,vote_average,vote_count,director_id,year,month,day
0,43597,237000000,150,2787965087,Avatar,7.2,11800,4762,2009,Dec,Thursday
1,43598,300000000,139,961000000,Pirates of the Caribbean: At World's End,6.9,4500,4763,2007,May,Saturday
2,43599,245000000,107,880674609,Spectre,6.3,4466,4764,2015,Oct,Monday
3,43600,250000000,112,1084939099,The Dark Knight Rises,7.6,9106,4765,2012,Jul,Monday
5,43602,258000000,115,890871626,Spider-Man 3,5.9,3576,4767,2007,May,Tuesday


#### Get group count

In [31]:
mov_df["director_id"].nunique()

199

In [32]:
mov_df.groupby("director_id").ngroups

199

#### Get all groups

In [33]:
tmp_df = pd.DataFrame(
    data=[
        ["A", 11],
        ["C", 63],
        ["B", 30],
        ["A", 28],
        ["A", 45],
        ["C", 79],
    ],
    columns=["col_1", "col_2"],
)
tmp_df

,col_1,col_2
0,A,11
1,C,63
2,B,30
3,A,28
4,A,45
5,C,79


In [34]:
tmp_df.groupby("col_1").groups

{'A': [0, 3, 4], 'B': [2], 'C': [1, 5]}

#### Extract single group

##### Solution #1

In [35]:
mask = dir_df["director_name"] == "Christopher Nolan"
nolan_id = dir_df[mask]["id"]

mov_df.groupby("director_id").get_group(nolan_id.item())

,id,budget,popularity,revenue,title,vote_average,vote_count,director_id,year,month,day
3,43600,250000000,112,1084939099,The Dark Knight Rises,7.6,9106,4765,2012,Jul,Monday
65,43662,185000000,187,1004558444,The Dark Knight,8.2,12002,4765,2008,Jul,Wednesday
95,43692,165000000,724,675120017,Interstellar,8.1,10867,4765,2014,Nov,Wednesday
96,43693,160000000,167,825532764,Inception,8.1,13752,4765,2010,Jul,Wednesday
119,43716,150000000,115,374218673,Batman Begins,7.5,7359,4765,2005,Jun,Friday
1033,44630,46000000,41,113714830,Insomnia,6.8,1148,4765,2002,May,Friday
1196,44793,40000000,74,109676311,The Prestige,8.0,4391,4765,2006,Oct,Thursday
3573,47170,9000000,60,39723096,Memento,8.1,4028,4765,2000,Oct,Wednesday


##### Solution #2

In [36]:
full_df.groupby("director_name").get_group("Christopher Nolan")

,id_x,budget,popularity,revenue,title,vote_average,vote_count,director_id,year,month,day,director_name,id_y,gender
3,43600,250000000,112,1084939099,The Dark Knight Rises,7.6,9106,4765,2012,Jul,Monday,Christopher Nolan,4765,Male
45,43662,185000000,187,1004558444,The Dark Knight,8.2,12002,4765,2008,Jul,Wednesday,Christopher Nolan,4765,Male
58,43692,165000000,724,675120017,Interstellar,8.1,10867,4765,2014,Nov,Wednesday,Christopher Nolan,4765,Male
59,43693,160000000,167,825532764,Inception,8.1,13752,4765,2010,Jul,Wednesday,Christopher Nolan,4765,Male
74,43716,150000000,115,374218673,Batman Begins,7.5,7359,4765,2005,Jun,Friday,Christopher Nolan,4765,Male
565,44630,46000000,41,113714830,Insomnia,6.8,1148,4765,2002,May,Friday,Christopher Nolan,4765,Male
641,44793,40000000,74,109676311,The Prestige,8.0,4391,4765,2006,Oct,Thursday,Christopher Nolan,4765,Male
1341,47170,9000000,60,39723096,Memento,8.1,4028,4765,2000,Oct,Wednesday,Christopher Nolan,4765,Male


### 3.2 Group based Aggregation

#### Get count of movies directed by each director

##### Solution #1

In [37]:
mov_df["director_id"].value_counts()

director_id
4799    26
4809    19
5087    19
5457    18
4779    16
        ..
5709     5
5894     5
5951     5
6056     5
6204     5
Name: count, Length: 199, dtype: int64

##### Solution #2

In [38]:
full_df.groupby("director_name")["title"].count()

director_name
Adam McKay                      6
Adam Shankman                   8
Alejandro González Iñárritu     6
Alex Proyas                     5
Alexander Payne                 5
                               ..
Wes Craven                     10
Wolfgang Petersen               7
Woody Allen                    18
Zack Snyder                     7
Zhang Yimou                     6
Name: title, Length: 199, dtype: int64

#### Get the first and latest year of direction for each director

In [39]:
full_df.groupby("director_name")["year"].aggregate(["min", "max", "count"])

,min,max,count
director_name,,,
Adam McKay,2004,2015,6
Adam Shankman,2001,2012,8
Alejandro González Iñárritu,2000,2015,6
Alex Proyas,1994,2016,5
Alexander Payne,1999,2013,5
...,...,...,...
Wes Craven,1984,2011,10
Wolfgang Petersen,1981,2006,7
Woody Allen,1977,2013,18


In [40]:
er_df = pd.DataFrame(
    data=[
        [1, 5.0],
        [2, 4.5],
        [2, 3.8],
        [1, 4.5],
        [1, 2.0],
    ],
    columns=["emp_id", "rating"],
)
er_df

,emp_id,rating
0,1,5.0
1,2,4.5
2,2,3.8
3,1,4.5
4,1,2.0


In [41]:
emp_1 = er_df["emp_id"] == 1
round(er_df[emp_1]["rating"].mean().item(), 2)

3.83

In [42]:
emp_1 = er_df["emp_id"] == 2
round(er_df[emp_1]["rating"].mean().item(), 2)

4.15

In [43]:
er_df.groupby("emp_id").mean()

,rating
emp_id,
1,3.833333
2,4.150000


In [44]:
er_df.groupby("emp_id").count()

,rating
emp_id,
1,3
2,2


### 3.3 Group based Filtering

#### Get movies of high budget directors

In [45]:
full_df["revenue_mil"] = full_df["revenue"] / 1_000_000
full_df["budget_mil"] = full_df["budget"] / 1_000_000

full_df.head()

,id_x,budget,popularity,revenue,title,vote_average,vote_count,director_id,year,month,day,director_name,id_y,gender,revenue_mil,budget_mil
0,43597,237000000,150,2787965087,Avatar,7.2,11800,4762,2009,Dec,Thursday,James Cameron,4762,Male,2787.965087,237.0
1,43598,300000000,139,961000000,Pirates of the Caribbean: At World's End,6.9,4500,4763,2007,May,Saturday,Gore Verbinski,4763,Male,961.000000,300.0
2,43599,245000000,107,880674609,Spectre,6.3,4466,4764,2015,Oct,Monday,Sam Mendes,4764,Male,880.674609,245.0
3,43600,250000000,112,1084939099,The Dark Knight Rises,7.6,9106,4765,2012,Jul,Monday,Christopher Nolan,4765,Male,1084.939099,250.0
4,43602,258000000,115,890871626,Spider-Man 3,5.9,3576,4767,2007,May,Tuesday,Sam Raimi,4767,Male,890.871626,258.0


##### Solution #1

###### STEP #1:

In [46]:
dir_bud = full_df.groupby("director_name")["budget_mil"].max().reset_index()
dir_bud.head()

,director_name,budget_mil
0,Adam McKay,100.0
1,Adam Shankman,80.0
2,Alejandro González Iñárritu,135.0
3,Alex Proyas,140.0
4,Alexander Payne,30.0


###### STEP #2:

In [47]:
high_bud_dir_fltr = dir_bud["budget_mil"] > 100
high_bud_dir = dir_bud[high_bud_dir_fltr]
high_bud_dir.head()

,director_name,budget_mil
2,Alejandro González Iñárritu,135.0
3,Alex Proyas,140.0
5,Andrew Adamson,225.0
10,Ang Lee,137.0
15,Barry Sonnenfeld,225.0


In [48]:
# print(sol_1.shape)
# sol_1.head()

##### Solution #2

In [49]:
high_bud_dir_fltr = (full_df.groupby("director_name")["budget_mil"].max() > 100).reset_index(
    drop=True
)
high_bud_dir = dir_bud[high_bud_dir_fltr]
high_bud_dir.head()

,director_name,budget_mil
2,Alejandro González Iñárritu,135.0
3,Alex Proyas,140.0
5,Andrew Adamson,225.0
10,Ang Lee,137.0
15,Barry Sonnenfeld,225.0


In [50]:
sol_2 = full_df[full_df["director_name"].isin(high_bud_dir["director_name"])]

print(sol_2.shape)
sol_2.head()

(636, 16)


,id_x,budget,popularity,revenue,title,vote_average,vote_count,director_id,year,month,day,director_name,id_y,gender,revenue_mil,budget_mil
0,43597,237000000,150,2787965087,Avatar,7.2,11800,4762,2009,Dec,Thursday,James Cameron,4762,Male,2787.965087,237.0
1,43598,300000000,139,961000000,Pirates of the Caribbean: At World's End,6.9,4500,4763,2007,May,Saturday,Gore Verbinski,4763,Male,961.000000,300.0
2,43599,245000000,107,880674609,Spectre,6.3,4466,4764,2015,Oct,Monday,Sam Mendes,4764,Male,880.674609,245.0
3,43600,250000000,112,1084939099,The Dark Knight Rises,7.6,9106,4765,2012,Jul,Monday,Christopher Nolan,4765,Male,1084.939099,250.0
4,43602,258000000,115,890871626,Spider-Man 3,5.9,3576,4767,2007,May,Tuesday,Sam Raimi,4767,Male,890.871626,258.0


##### Solution #3

In [51]:
sol_3 = full_df.groupby("director_name").filter(lambda grp: grp["budget_mil"].max() > 100)
print(sol_3.shape)
sol_3.head()

(636, 16)


,id_x,budget,popularity,revenue,title,vote_average,vote_count,director_id,year,month,day,director_name,id_y,gender,revenue_mil,budget_mil
0,43597,237000000,150,2787965087,Avatar,7.2,11800,4762,2009,Dec,Thursday,James Cameron,4762,Male,2787.965087,237.0
1,43598,300000000,139,961000000,Pirates of the Caribbean: At World's End,6.9,4500,4763,2007,May,Saturday,Gore Verbinski,4763,Male,961.000000,300.0
2,43599,245000000,107,880674609,Spectre,6.3,4466,4764,2015,Oct,Monday,Sam Mendes,4764,Male,880.674609,245.0
3,43600,250000000,112,1084939099,The Dark Knight Rises,7.6,9106,4765,2012,Jul,Monday,Christopher Nolan,4765,Male,1084.939099,250.0
4,43602,258000000,115,890871626,Spider-Man 3,5.9,3576,4767,2007,May,Tuesday,Sam Raimi,4767,Male,890.871626,258.0


### 3.4 Group based Apply

#### Get movies with profit greater than avg revenue of directors

In [52]:
full_df.groupby("director_name").filter(lambda grp: grp["budget"].max() > grp["revenue"].mean())

,id_x,budget,popularity,revenue,title,vote_average,vote_count,director_id,year,month,day,director_name,id_y,gender,revenue_mil,budget_mil
7,43608,200000000,107,586090727,Quantum of Solace,6.1,2965,4773,2008,Oct,Thursday,Marc Forster,4773,Male,586.090727,200.000
12,43614,380000000,135,1045713802,Pirates of the Caribbean: On Stranger Tides,6.4,4948,4775,2011,May,Saturday,Rob Marshall,4775,Male,1045.713802,380.000
15,43618,200000000,37,310669540,Robin Hood,6.2,1398,4779,2010,May,Wednesday,Ridley Scott,4779,Male,310.669540,200.000
20,43624,209000000,64,303025485,Battleship,5.5,2114,4782,2012,Apr,Wednesday,Peter Berg,4782,Male,303.025485,209.000
24,43630,210000000,3,459359555,X-Men: The Last Stand,6.3,3525,4786,2006,May,Wednesday,Brett Ratner,4786,Male,459.359555,210.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1459,48359,0,2,0,George Washington,6.4,36,5231,2000,Oct,Sunday,David Gordon Green,5231,Male,0.000000,0.000
1460,48363,0,3,321952,The Last Waltz,7.9,64,4809,1978,May,Monday,Martin Scorsese,4809,Male,0.321952,0.000
1461,48370,27000,19,3151130,Clerks,7.4,755,5369,1994,Sep,Tuesday,Kevin Smith,5369,Male,3.151130,0.027
1462,48375,0,7,0,Rampage,6.0,131,5148,2009,Aug,Friday,Uwe Boll,5148,Male,0.000000,0.000


In [53]:
def risky_movie(x):
    # returns whether a movies is risky or not
    # print(data.get_group(x))
    # print(x)
    x["risky"] = x["budget"] - x["revenue"].mean() >= 0
    return x


data_risky = full_df.groupby("director_name").apply(risky_movie)
data_risky

id_x     budget  popularity    revenue  \
director_name                                                 
Adam McKay    176   43882  100000000          24  170432927   
              323   44151   72500000          12  162966177   
              366   44236   65000000          22  128107642   
              505   44503   50000000          38  173649015   
              839   45301   28000000          57  133346506   
...                   ...        ...         ...        ...   
Zhang Yimou   590   44692        110           9          0   
              604   44733   31000000          23  177394432   
              1217  46460          0          21   92863945   
              1223  46493          0           1          0   
              1389  47489          0           6          0   

                                                          title  vote_average  \
director_name                                                                   
Adam McKay    176                                The Other Guys           6.1   
              323   Talladega Nights: The Ballad of Ricky Bobby           6.2   
              366                                 Step Brothers           6.5   
              505             Anchorman 2: The Legend Continues           6.0   
              839                                 The Big Short           7.3   
...                                                         ...           ...   
Zhang Yimou   590                    Curse of the Golden Flower           6.6   
              604                                          Hero           7.2   
              1217                      House of Flying Daggers           7.1   
              1223             A Woman, a Gun and a Noodle Shop           4.8   
              1389                                  Coming Home           6.9   

                    vote_count  director_id  year month        day  id_y  \
director_name                                                              
Adam McKay    176         1383         4925  2010   Aug     Friday  4925   
              323          491         4925  2006   Aug     Friday  4925   
              366         1062         4925  2008   Jul     Friday  4925   
              505          923         4925  2013   Dec  Wednesday  4925   
              839         2607         4925  2015   Dec     Friday  4925   
...                        ...          ...   ...   ...        ...   ...   
Zhang Yimou   590          203         4945  2006   Dec   Thursday  4945   
              604          635         4945  2002   Dec   Thursday  4945   
              1217         439         4945  2004   May  Wednesday  4945   
              1223          13         4945  2009   Dec     Friday  4945   
              1389          49         4945  2014   May     Friday  4945   

                   gender  revenue_mil  budget_mil  risky  
director_name                                              
Adam McKay    176    Male   170.432927   100.00000  False  
              323    Male   162.966177    72.50000  False  
              366    Male   128.107642    65.00000  False  
              505    Male   173.649015    50.00000  False  
              839    Male   133.346506    28.00000  False  
...                   ...          ...         ...    ...  
Zhang Yimou   590    Male     0.000000     0.00011  False  
              604    Male   177.394432    31.00000  False  
              1217   Male    92.863945     0.00000  False  
              1223   Male     0.000000     0.00000  False  
              1389   Male     0.000000     0.00000  False  

[1465 rows x 16 columns]

In [54]:
data_risky.loc[data_risky["risky"]]

id_x     budget  popularity    revenue  \
director_name                                                      
Andrzej Bartkowiak 349   44192   60000000          29   55987321   
Atom Egoyan        946   45538   25000000           4          0   
                   1194  46370   15000000          26    8459458   
                   1347  47224    5000000           7    3263585   
Brett Ratner       24    43630  210000000           3  459359555   
...                        ...        ...         ...        ...   
Uwe Boll           944   45536   25000000           7    2405420   
                   1058  45834   20000000           9   10442808   
                   1383  47453    3500000           4          0   
Wayne Wang         468   44419   55000000          19  154906693   
Zhang Yimou        192   43914   94000000          12   95311434   

                                               title  vote_average  \
director_name                                                        
Andrzej Bartkowiak 349                          Doom           5.0   
Atom Egoyan        946          Where the Truth Lies           5.9   
                   1194                        Chloe           5.9   
                   1347          The Sweet Hereafter           6.8   
Brett Ratner       24          X-Men: The Last Stand           6.3   
...                                              ...           ...   
Uwe Boll           944                    BloodRayne           3.5   
                   1058            Alone in the Dark           3.1   
                   1383  In the Name of the King III           3.3   
Wayne Wang         468             Maid in Manhattan           5.6   
Zhang Yimou        192            The Flowers of War           7.1   

                         vote_count  director_id  year month        day  id_y  \
director_name                                                                   
Andrzej Bartkowiak 349          609         5061  2005   Oct   Thursday  5061   
Atom Egoyan        946           66         5599  2005   Oct     Friday  5599   
                   1194         498         5599  2009   Mar  Wednesday  5599   
                   1347         103         5599  1997   May  Wednesday  5599   
Brett Ratner       24          3525         4786  2006   May  Wednesday  4786   
...                             ...          ...   ...   ...        ...   ...   
Uwe Boll           944          118         5148  2005   Oct   Saturday  5148   
                   1058         173         5148  2005   Jan     Friday  5148   
                   1383          19         5148  2013   Dec     Friday  5148   
Wayne Wang         468          485         5162  2002   Dec     Friday  5162   
Zhang Yimou        192          187         4945  2011   Dec   Thursday  4945   

                        gender  revenue_mil  budget_mil  risky  
director_name                                                   
Andrzej Bartkowiak 349    Male    55.987321        60.0   True  
Atom Egoyan        946    Male     0.000000        25.0   True  
                   1194   Male     8.459458        15.0   True  
                   1347   Male     3.263585         5.0   True  
Brett Ratner       24     Male   459.359555       210.0   True  
...                        ...          ...         ...    ...  
Uwe Boll           944    Male     2.405420        25.0   True  
                   1058   Male    10.442808        20.0   True  
                   1383   Male     0.000000         3.5   True  
Wayne Wang         468     NaN   154.906693        55.0   True  
Zhang Yimou        192    Male    95.311434        94.0   True  

[131 rows x 16 columns]

## 4 Usecase

### 4.1 Group basics

#### Test dataset

In [55]:
dm_df = pd.DataFrame(
    [
        ["A", "MA1", 30, 25],
        ["C", "MC1", 50, 55],
        ["B", "MB1", 30, 32],
        ["B", "MB2", 35, 40],
        ["C", "MC3", 200, 190],
        ["A", "MA2", 80, 90],
        ["C", "MC2", 20, 50],
        ["C", "MC3", 80, 120],
        ["A", "MA3", 160, 100],
    ],
    columns=["director", "movie", "budget", "revenue"],
)
dm_df

,director,movie,budget,revenue
0,A,MA1,30,25
1,C,MC1,50,55
2,B,MB1,30,32
3,B,MB2,35,40
4,C,MC3,200,190
5,A,MA2,80,90
6,C,MC2,20,50
7,C,MC3,80,120
8,A,MA3,160,100


In [56]:
dm_df.groupby("director").ngroups

3

In [57]:
dm_df.groupby("director").groups

{'A': [0, 5, 8], 'B': [2, 3], 'C': [1, 4, 6, 7]}

### 4.2 Group based Aggregation

#### Count

In [58]:
dm_df.groupby("director").count()

,movie,budget,revenue
director,,,
A,3,3,3
B,2,2,2
C,4,4,4


#### Mean

In [59]:
dm_df.groupby("director").agg({"budget": "mean"})

,budget
director,
A,90.0
B,32.5
C,87.5


In [60]:
dm_df.groupby("director")["revenue"].mean()

director
A     71.666667
B     36.000000
C    103.750000
Name: revenue, dtype: float64

#### Multi column aggregation

In [61]:
dm_df.groupby("director").aggregate(
    {
        "movie": ["count"],
        "budget": ["min", "max", "mean"],
        "revenue": ["min", "max", "mean"],
    },
)

movie budget            revenue                 
         count    min  max  mean     min  max        mean
director                                                 
A            3     30  160  90.0      25  100   71.666667
B            2     30   35  32.5      32   40   36.000000
C            4     20  200  87.5      50  190  103.750000

In [62]:
dm_df

,director,movie,budget,revenue
0,A,MA1,30,25
1,C,MC1,50,55
2,B,MB1,30,32
3,B,MB2,35,40
4,C,MC3,200,190
5,A,MA2,80,90
6,C,MC2,20,50
7,C,MC3,80,120
8,A,MA3,160,100


In [63]:
dm_df.groupby("director").aggregate(
    movie_count=("movie", "count"),
    budget_min=("budget", "min"),
    budget_max=("budget", "max"),
    budget_mean=("budget", "mean"),
    revenue_min=("revenue", "min"),
    revenue_max=("revenue", "max"),
    revenue_mean=("revenue", "mean"),
)

,movie_count,budget_min,budget_max,budget_mean,revenue_min,revenue_max,revenue_mean
director,,,,,,,
A,3,30,160,90.0,25,100,71.666667
B,2,30,35,32.5,32,40,36.000000
C,4,20,200,87.5,50,190,103.750000


In [64]:
dm_df.groupby("director").describe()

budget                                                    revenue  \
          count  mean        std   min    25%   50%     75%    max   count   
director                                                                     
A           3.0  90.0  65.574385  30.0  55.00  80.0  120.00  160.0     3.0   
B           2.0  32.5   3.535534  30.0  31.25  32.5   33.75   35.0     2.0   
C           4.0  87.5  78.898669  20.0  42.50  65.0  110.00  200.0     4.0   

                                                                  
                mean        std   min    25%   50%    75%    max  
director                                                          
A          71.666667  40.722639  25.0  57.50  90.0   95.0  100.0  
B          36.000000   5.656854  32.0  34.00  36.0   38.0   40.0  
C         103.750000  65.748891  50.0  53.75  87.5  137.5  190.0

### 4.3 Group based Filtering

In [65]:
dm_df

,director,movie,budget,revenue
0,A,MA1,30,25
1,C,MC1,50,55
2,B,MB1,30,32
3,B,MB2,35,40
4,C,MC3,200,190
5,A,MA2,80,90
6,C,MC2,20,50
7,C,MC3,80,120
8,A,MA3,160,100


#### Get movies of high budget directors

##### Solution #1

In [66]:
dm_df.groupby("director")["budget"].max()

director
A    160
B     35
C    200
Name: budget, dtype: int64

In [67]:
data_dir_budget = dm_df.groupby("director")["budget"].max().reset_index()
data_dir_budget

,director,budget
0,A,160
1,B,35
2,C,200


In [68]:
names = data_dir_budget.loc[data_dir_budget["budget"] >= 100, "director"]
names

0    A
2    C
Name: director, dtype: str

In [69]:
dm_df.loc[dm_df["director"].isin(names)]

,director,movie,budget,revenue
0,A,MA1,30,25
1,C,MC1,50,55
4,C,MC3,200,190
5,A,MA2,80,90
6,C,MC2,20,50
7,C,MC3,80,120
8,A,MA3,160,100


##### Solution #2

In [70]:
dm_df.groupby("director").filter(lambda grp: grp["budget"].max() > 100)

,director,movie,budget,revenue
0,A,MA1,30,25
1,C,MC1,50,55
4,C,MC3,200,190
5,A,MA2,80,90
6,C,MC2,20,50
7,C,MC3,80,120
8,A,MA3,160,100


#### Get movies with profit greater than avg revenue of directors

##### Solution #1

In [71]:
def risky_movies(grp):
    grp["is_risky"] = grp["budget"] - grp["revenue"].mean() > 0
    return grp


dm_df.groupby("director").apply(risky_movies)

movie  budget  revenue  is_risky
director                                   
A        0   MA1      30       25     False
         5   MA2      80       90      True
         8   MA3     160      100      True
B        2   MB1      30       32     False
         3   MB2      35       40     False
C        1   MC1      50       55     False
         4   MC3     200      190      True
         6   MC2      20       50     False
         7   MC3      80      120     False

In [72]:
def risky_movie(x):
    # returns whether a movies is risky or not
    # print(data.get_group(x))
    # print(x)
    x["risky"] = x["budget"] - x["revenue"].mean() >= 0
    return x


fltr = dm_df.groupby("director").apply(lambda row: row["budget"] - row["revenue"].mean() > 0)
fltr

director   
A         0    False
          5     True
          8     True
B         2    False
          3    False
C         1    False
          4     True
          6    False
          7    False
Name: budget, dtype: bool

In [73]:
def risky_movie(x):
    # returns whether a movies is risky or not
    # print(data.get_group(x))
    # print(x)
    x["risky"] = x["budget"] - x["revenue"].mean() >= 0
    return x


data_risky = full_df.groupby("director_name").apply(risky_movie)
data_risky

id_x     budget  popularity    revenue  \
director_name                                                 
Adam McKay    176   43882  100000000          24  170432927   
              323   44151   72500000          12  162966177   
              366   44236   65000000          22  128107642   
              505   44503   50000000          38  173649015   
              839   45301   28000000          57  133346506   
...                   ...        ...         ...        ...   
Zhang Yimou   590   44692        110           9          0   
              604   44733   31000000          23  177394432   
              1217  46460          0          21   92863945   
              1223  46493          0           1          0   
              1389  47489          0           6          0   

                                                          title  vote_average  \
director_name                                                                   
Adam McKay    176                                The Other Guys           6.1   
              323   Talladega Nights: The Ballad of Ricky Bobby           6.2   
              366                                 Step Brothers           6.5   
              505             Anchorman 2: The Legend Continues           6.0   
              839                                 The Big Short           7.3   
...                                                         ...           ...   
Zhang Yimou   590                    Curse of the Golden Flower           6.6   
              604                                          Hero           7.2   
              1217                      House of Flying Daggers           7.1   
              1223             A Woman, a Gun and a Noodle Shop           4.8   
              1389                                  Coming Home           6.9   

                    vote_count  director_id  year month        day  id_y  \
director_name                                                              
Adam McKay    176         1383         4925  2010   Aug     Friday  4925   
              323          491         4925  2006   Aug     Friday  4925   
              366         1062         4925  2008   Jul     Friday  4925   
              505          923         4925  2013   Dec  Wednesday  4925   
              839         2607         4925  2015   Dec     Friday  4925   
...                        ...          ...   ...   ...        ...   ...   
Zhang Yimou   590          203         4945  2006   Dec   Thursday  4945   
              604          635         4945  2002   Dec   Thursday  4945   
              1217         439         4945  2004   May  Wednesday  4945   
              1223          13         4945  2009   Dec     Friday  4945   
              1389          49         4945  2014   May     Friday  4945   

                   gender  revenue_mil  budget_mil  risky  
director_name                                              
Adam McKay    176    Male   170.432927   100.00000  False  
              323    Male   162.966177    72.50000  False  
              366    Male   128.107642    65.00000  False  
              505    Male   173.649015    50.00000  False  
              839    Male   133.346506    28.00000  False  
...                   ...          ...         ...    ...  
Zhang Yimou   590    Male     0.000000     0.00011  False  
              604    Male   177.394432    31.00000  False  
              1217   Male    92.863945     0.00000  False  
              1223   Male     0.000000     0.00000  False  
              1389   Male     0.000000     0.00000  False  

[1465 rows x 16 columns]